In [ ]:
# Imports

import os
import json
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Paths
BASE = "/content/drive/MyDrive/HDA-IWBF-2023/synthetic-tattoo-images"

IMAGE_PATH = os.path.join(BASE, "JPGImages")
MASK_PATH  = os.path.join(BASE, "SegmentationClass")

FEATURES_OUT_DIR = os.path.join(BASE, "features_batches")
os.makedirs(FEATURES_OUT_DIR, exist_ok=True)


In [ ]:
# Batch size
BATCH_SIZE = 500
BATCH_INDEX = 0

In [ ]:
# Output paths
BATCH_JSON_OUT = os.path.join(
    FEATURES_OUT_DIR,
    f"features_batch_{BATCH_INDEX:02d}.json"
)

BATCH_CSV_OUT = os.path.join(
    FEATURES_OUT_DIR,
    f"features_batch_{BATCH_INDEX:02d}.csv"
)

BATCH_ERRORS_OUT = os.path.join(
    FEATURES_OUT_DIR,
    f"errors_batch_{BATCH_INDEX:02d}.json"
)

In [ ]:
# IO helpers
def read_image(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_COLOR)

    if img is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {path}")

    return img


def read_mask(path: str) -> np.ndarray:
    mask = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if mask is None:
        raise FileNotFoundError(f"No se pudo leer la máscara: {path}")

    if len(mask.shape) == 3:
        mask = mask[:, :, 0]

    return mask


def list_pairs(image_dir: str, mask_dir: str):
    imgs = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    pairs = []

    for f in imgs:
        base_name = os.path.splitext(f)[0]

        candidates = [
            base_name + ".png",
            base_name + ".jpg",
            base_name + ".jpeg"
        ]

        mask_path = None

        for cand in candidates:
            candidate_path = os.path.join(mask_dir, cand)

            if os.path.exists(candidate_path):
                mask_path = candidate_path
                break

        if mask_path is not None:
            image_path = os.path.join(image_dir, f)
            pairs.append((image_path, mask_path))

    return pairs


def safe_div(a, b):
    return float(a) / float(b) if b != 0 else 0.0


def sanitize_for_json(obj):
    if isinstance(obj, dict):
        return {k: sanitize_for_json(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [sanitize_for_json(v) for v in obj]

    if isinstance(obj, tuple):
        return [sanitize_for_json(v) for v in obj]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    return obj

In [ ]:
# Position helper
def get_position_label(cx: float, cy: float) -> str:
    if cx < 0.33:
        horiz = "left"
    elif cx > 0.66:
        horiz = "right"
    else:
        horiz = "center"

    if cy < 0.33:
        vert = "top"
    elif cy > 0.66:
        vert = "bottom"
    else:
        vert = "center"

    if vert == "center" and horiz == "center":
        return "center"
    elif vert == "center":
        return f"center-{horiz}"
    elif horiz == "center":
        return f"{vert}-center"
    else:
        return f"{vert}-{horiz}"

In [ ]:
# Mask parsing
def mask_to_regions(mask: np.ndarray):
    if len(mask.shape) == 3:
        mask = mask[:, :, 0]

    background = (mask <= 30).astype(np.uint8)
    skin = (mask >= 220).astype(np.uint8)
    tattoo = ((mask > 30) & (mask < 220)).astype(np.uint8)

    return tattoo, skin, background

In [ ]:
# Geometry helpers
def bbox_from_mask(bin_mask: np.ndarray):
    ys, xs = np.where(bin_mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def expand_bbox(bbox, img_shape, margin_ratio=0.08):
    if bbox is None:
        return None

    x0, y0, x1, y1 = bbox
    h, w = img_shape[:2]

    bw = x1 - x0 + 1
    bh = y1 - y0 + 1

    mx = int(bw * margin_ratio)
    my = int(bh * margin_ratio)

    x0e = max(0, x0 - mx)
    y0e = max(0, y0 - my)
    x1e = min(w - 1, x1 + mx)
    y1e = min(h - 1, y1 + my)

    return x0e, y0e, x1e, y1e


def crop_from_bbox(img: np.ndarray, bbox):
    if bbox is None:
        return None

    x0, y0, x1, y1 = bbox
    return img[y0:y1 + 1, x0:x1 + 1].copy()


def remove_small_components(mask_bin: np.ndarray, min_area_ratio=0.01, min_area_px=30):
    total_area = int(mask_bin.sum())

    if total_area == 0:
        return mask_bin.copy()

    nlabels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_bin,
        connectivity=8
    )

    threshold = max(min_area_px, int(total_area * min_area_ratio))
    clean_mask = np.zeros_like(mask_bin, dtype=np.uint8)

    for i in range(1, nlabels):
        area = int(stats[i, cv2.CC_STAT_AREA])

        if area >= threshold:
            clean_mask[labels == i] = 1

    return clean_mask

In [ ]:
# Features extraction
def extract_features_structured(img_bgr: np.ndarray, mask_gray: np.ndarray, filename: str):
    raw_tattoo, skin, background = mask_to_regions(mask_gray)

    clean_tattoo = remove_small_components(
        raw_tattoo,
        min_area_ratio=0.01,
        min_area_px=30
    )

    h, w = mask_gray.shape[:2]

    tattoo_area = int(clean_tattoo.sum())
    skin_area = int(skin.sum())

    bbox = bbox_from_mask(clean_tattoo)
    bbox_exp = expand_bbox(bbox, img_bgr.shape, margin_ratio=0.08) if bbox is not None else None
    crop_img = crop_from_bbox(img_bgr, bbox_exp) if bbox_exp is not None else None

    result = {
        "metadata": {
            "filename": filename,
            "image_height": int(h),
            "image_width": int(w),
        },
        "spatial_features": {},
        "shape_features": {},
        "appearance_features": {},
    }

    if tattoo_area == 0:
        result["spatial_features"] = {
            "tattoo_area_px": 0,
            "skin_area_px": int(skin_area),
            "tattoo_area_frac_img": 0.0,
            "tattoo_over_skin_frac": 0.0,
            "bbox": None,
            "bbox_aspect_ratio": 0.0,
            "position": {
                "x_norm": -1.0,
                "y_norm": -1.0,
                "label": "none"
            }
        }

        result["shape_features"] = {
            "perimeter": 0.0,
            "circularity": 0.0,
            "solidity": 0.0,
            "ellipse_major_axis": 0.0,
            "ellipse_minor_axis": 0.0,
            "elongation": 0.0,
            "orientation_deg": 0.0,
        }

        result["appearance_features"] = {
            "tattoo_gray_mean": None,
            "tattoo_gray_std": None,
            "skin_gray_mean": None,
            "skin_gray_std": None,
            "contrast_skin_minus_tattoo": None,
            "cohens_d": None,
            "crop_gray_mean": None,
            "crop_gray_std": None,
            "edge_density_crop": None,
            "entropy_crop": None,
        }

        return result

    ys, xs = np.where(clean_tattoo > 0)

    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())

    bw = x1 - x0 + 1
    bh = y1 - y0 + 1

    cx = float(xs.mean()) / w
    cy = float(ys.mean()) / h

    result["spatial_features"] = {
        "tattoo_area_px": int(tattoo_area),
        "skin_area_px": int(skin_area),
        "tattoo_area_frac_img": safe_div(tattoo_area, h * w),
        "tattoo_over_skin_frac": safe_div(tattoo_area, skin_area),
        "bbox": {
            "x": int(x0),
            "y": int(y0),
            "w": int(bw),
            "h": int(bh),
        },
        "bbox_aspect_ratio": safe_div(bw, bh),
        "position": {
            "x_norm": cx,
            "y_norm": cy,
            "label": get_position_label(cx, cy)
        }
    }

    contours, _ = cv2.findContours(
        clean_tattoo,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    perimeter = 0.0
    circularity = 0.0
    solidity = 0.0
    ellipse_major = 0.0
    ellipse_minor = 0.0
    elongation = 0.0
    orientation_deg = 0.0

    if contours:
        cnt = max(contours, key=cv2.contourArea)

        area = float(cv2.contourArea(cnt))
        perimeter = float(cv2.arcLength(cnt, True))
        circularity = safe_div(4 * np.pi * area, perimeter * perimeter)

        hull = cv2.convexHull(cnt)
        hull_area = float(cv2.contourArea(hull))
        solidity = safe_div(area, hull_area)

        if len(cnt) >= 5:
            (_, _), (MA, ma), angle = cv2.fitEllipse(cnt)

            ellipse_major = float(max(MA, ma))
            ellipse_minor = float(min(MA, ma))
            elongation = safe_div(ellipse_major, ellipse_minor)
            orientation_deg = float(angle)

    result["shape_features"] = {
        "perimeter": perimeter,
        "circularity": circularity,
        "solidity": solidity,
        "ellipse_major_axis": ellipse_major,
        "ellipse_minor_axis": ellipse_minor,
        "elongation": elongation,
        "orientation_deg": orientation_deg,
    }

    gray_full = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    tattoo_pixels = gray_full[clean_tattoo > 0]
    skin_pixels = gray_full[skin > 0] if skin.sum() > 0 else None

    tattoo_gray_mean = float(tattoo_pixels.mean())
    tattoo_gray_std = float(tattoo_pixels.std())

    skin_gray_mean = None
    skin_gray_std = None
    contrast_skin_minus_tattoo = None
    cohens_d = None

    if skin_pixels is not None and len(skin_pixels) > 50:
        skin_gray_mean = float(skin_pixels.mean())
        skin_gray_std = float(skin_pixels.std())

        diff = skin_gray_mean - tattoo_gray_mean
        pooled = np.sqrt((skin_gray_std ** 2 + tattoo_gray_std ** 2) / 2.0)

        contrast_skin_minus_tattoo = float(diff)
        cohens_d = safe_div(diff, pooled)

    crop_gray_mean = None
    crop_gray_std = None
    edge_density_crop = None
    entropy_crop = None

    if crop_img is not None:
        gray_crop = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)

        crop_gray_mean = float(gray_crop.mean())
        crop_gray_std = float(gray_crop.std())

        edges_crop = cv2.Canny(gray_crop, 80, 160)
        edge_density_crop = float(edges_crop.mean() / 255.0)

        hist = cv2.calcHist([gray_crop], [0], None, [256], [0, 256]).ravel()
        p = hist / (hist.sum() + 1e-9)
        entropy_crop = float(-np.sum(p[p > 0] * np.log2(p[p > 0])))

    result["appearance_features"] = {
        "tattoo_gray_mean": tattoo_gray_mean,
        "tattoo_gray_std": tattoo_gray_std,
        "skin_gray_mean": skin_gray_mean,
        "skin_gray_std": skin_gray_std,
        "contrast_skin_minus_tattoo": contrast_skin_minus_tattoo,
        "cohens_d": cohens_d,
        "crop_gray_mean": crop_gray_mean,
        "crop_gray_std": crop_gray_std,
        "edge_density_crop": edge_density_crop,
        "entropy_crop": entropy_crop,
    }

    return result

In [ ]:
# Process selected batch
pairs = list_pairs(IMAGE_PATH, MASK_PATH)

print("Total pares imagen-máscara encontrados:", len(pairs))

if not pairs:
    raise RuntimeError("No se encontraron pares imagen-máscara. Revisa las rutas.")

start_idx = BATCH_INDEX * BATCH_SIZE
end_idx = min(start_idx + BATCH_SIZE, len(pairs))

batch_pairs = pairs[start_idx:end_idx]

print(f"Procesando batch {BATCH_INDEX}")
print(f"Rango de imágenes: {start_idx} - {end_idx - 1}")
print(f"Número de imágenes en este batch: {len(batch_pairs)}")

if not batch_pairs:
    raise RuntimeError("Este batch está vacío. Baja el valor de BATCH_INDEX.")

batch_features = []
batch_errors = []

for ipath, mpath in tqdm(batch_pairs, desc=f"Extracting batch {BATCH_INDEX}"):
    filename = os.path.basename(ipath)

    try:
        img = read_image(ipath)
        mask = read_mask(mpath)

        features = extract_features_structured(
            img_bgr=img,
            mask_gray=mask,
            filename=filename
        )

        features_clean = sanitize_for_json(features)
        batch_features.append(features_clean)

    except Exception as e:
        batch_errors.append({
            "image": os.path.basename(ipath),
            "mask": os.path.basename(mpath),
            "error": str(e)
        })

In [ ]:
# Save batch JSON
with open(BATCH_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(batch_features, f, indent=2, ensure_ascii=False)

In [ ]:
# Save batch CSV
df_batch = pd.json_normalize(batch_features)

if "spatial_features.bbox" in df_batch.columns:
    df_batch = df_batch.drop(columns=["spatial_features.bbox"])

df_batch.to_csv(BATCH_CSV_OUT, index=False)

In [ ]:
# Save batch errors
with open(BATCH_ERRORS_OUT, "w", encoding="utf-8") as f:
    json.dump(batch_errors, f, indent=2, ensure_ascii=False)

In [ ]:
# Summary
print("\n================ BATCH EXTRACTION SUMMARY ================\n")
print("Batch procesado:", BATCH_INDEX)
print("Rango procesado:", start_idx, "-", end_idx - 1)
print("Imágenes procesadas correctamente:", len(batch_features))
print("Errores:", len(batch_errors))
print("JSON del batch guardado en:", BATCH_JSON_OUT)
print("CSV del batch guardado en:", BATCH_CSV_OUT)
print("Errores del batch guardados en:", BATCH_ERRORS_OUT)

df_batch.head()

CELDA PARA UNIR TODOS LOS LOTES

In [ ]:
BATCHES_DIR = os.path.join(BASE, "features_batches")

MERGED_DIR = os.path.join(BASE, "features_merged")
os.makedirs(MERGED_DIR, exist_ok=True)


# Output files
MERGED_CSV_OUT = os.path.join(MERGED_DIR, "features_all.csv")
MERGED_JSON_OUT = os.path.join(MERGED_DIR, "features_all.json")


# Find batch files
csv_files = sorted([
    os.path.join(BATCHES_DIR, f)
    for f in os.listdir(BATCHES_DIR)
    if f.startswith("features_batch_") and f.endswith(".csv")
])

json_files = sorted([
    os.path.join(BATCHES_DIR, f)
    for f in os.listdir(BATCHES_DIR)
    if f.startswith("features_batch_") and f.endswith(".json")
])


print("CSV batches encontrados:", len(csv_files))
print("JSON batches encontrados:", len(json_files))


# Merge CSV
csv_dfs = []

for csv_path in csv_files:
    print("Leyendo CSV:", os.path.basename(csv_path))

    df = pd.read_csv(csv_path)
    csv_dfs.append(df)

df_all = pd.concat(csv_dfs, ignore_index=True)

df_all.to_csv(MERGED_CSV_OUT, index=False)

print("\nCSV global guardado en:")
print(MERGED_CSV_OUT)

print("Total filas CSV:", len(df_all))


# Merge JSON
all_json_features = []

for json_path in json_files:
    print("Leyendo JSON:", os.path.basename(json_path))

    with open(json_path, "r", encoding="utf-8") as f:
        batch_data = json.load(f)

    all_json_features.extend(batch_data)

with open(MERGED_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(all_json_features, f, indent=2, ensure_ascii=False)

print("\nJSON global guardado en:")
print(MERGED_JSON_OUT)

print("Total entradas JSON:", len(all_json_features))


# Final summary
print("\n================ MERGE COMPLETED ================\n")

print("CSV final:")
print(MERGED_CSV_OUT)

print("\nJSON final:")
print(MERGED_JSON_OUT)

print("\nNúmero total de muestras:")
print(len(df_all))